## Portable path configuration
This release copy contains no personal or NAS paths. Set `GAO2026_DATA_ROOT` for deposited data and `GAO2026_RAW_ROOT` only when running raw-image stages (raw revision microscopy is not included).


In [ ]:
from pathlib import Path
import os
_default_repo_root = Path.cwd().resolve()
GAO2026_DATA_ROOT = Path(os.environ.get('GAO2026_DATA_ROOT', _default_repo_root / 'data')).expanduser().resolve()
GAO2026_RAW_ROOT = Path(os.environ.get('GAO2026_RAW_ROOT', GAO2026_DATA_ROOT / 'external_raw')).expanduser().resolve()
GAO2026_OUTPUT_ROOT = Path(os.environ.get('GAO2026_OUTPUT_ROOT', _default_repo_root / 'outputs')).expanduser().resolve()
GAO2026_EXTERNAL_ROOT = Path(os.environ.get('GAO2026_EXTERNAL_ROOT', GAO2026_DATA_ROOT / 'external')).expanduser().resolve()


# SOX2 temporal-resolution cross-correlation analysis

This notebook contains the minimal analysis needed to compare SOX2 accumulation with nascent transcription at 30-s, 1-min, and 2-min sampling intervals. It is restricted to SOX2 data; HDAC3 and other factors are intentionally excluded.

The analysis follows the plotting and statistical conventions used in `analyses/02_Cross_correlation/notebooks/2_Timelapse_analysis.ipynb`:

- no interpolation of missing signal values;
- per-cell z-scoring of detrended SOX2/SNAP and MCP signals;
- pair-count-weighted group correlations with baseline correction;
- bootstrap 95% confidence intervals;
- lead-lag index (LLI) and time-shift permutation testing.

Lag convention: **negative lag means SOX2/SNAP leads MCP**. A positive raw LLI therefore means relatively stronger correlation on the SOX2-leading side.

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import wilcoxon

mpl.rcParams.update({
    "font.family": "Arial",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.dpi": 150,
})

DATASETS = [
    {
        "label": "SOX2_30s",
        "display_label": "30 s",
        "dt_min": 0.5,
        "csv_path": Path(
            str(GAO2026_RAW_ROOT / 'Public-TS-873-2/Microscope/Gao/260318-timelapse-SOX2/outputs/1_analysis/6_transcription_summary/all_cells_spot_intensity_seg_cell.csv')
        ),
    },
    {
        "label": "SOX2_1min",
        "display_label": "1 min",
        "dt_min": 1.0,
        "csv_path": Path(
            str(GAO2026_RAW_ROOT / 'Public-TS-873-2/Microscope/Gao/251111-timelapse-sox2/outputs/1_analysis/6_transcription_summary/all_cells_spot_intensity_seg_cell.csv')
        ),
    },
    {
        "label": "SOX2_2min",
        "display_label": "2 min",
        "dt_min": 2.0,
        "csv_path": Path(
            str(GAO2026_RAW_ROOT / 'Public-TS-873-2/Microscope/Ochiai/250704-live3-timelapse/for_analysis/seg_cell/output/all_cells_spot_intensity_seg_cell.csv')
        ),
    },
]

CONFIG = {
    "cell_min_points": 8,
    "pair_thresh": 3,
    "max_lag": 10,
    "base_lag_range": (7, 10),
    "use_zscore": True,
    "use_multitau_smoothing": True,
    "multitau_scheme": ((0, 3, 1), (4, 7, 3), (8, 10, 5)),
    "boot_n": 3000,
    "curve_boot_n": 2000,
    "perm_n": 1000,
    "lli_window_frames": 5,
    "seed": 12345,
}

LAGS = np.arange(-CONFIG["max_lag"], CONFIG["max_lag"] + 1, dtype=int)
OUTPUT_ROOT = Path("outputs")
OUTPUT_A = OUTPUT_ROOT / "output_A"
OUTPUT_LLI = OUTPUT_ROOT / "output_LLI"
OUTPUT_SUMMARY = OUTPUT_ROOT / "output_summary"
for output_dir in (OUTPUT_A, OUTPUT_LLI, OUTPUT_SUMMARY):
    output_dir.mkdir(parents=True, exist_ok=True)

missing_inputs = [str(item["csv_path"]) for item in DATASETS if not item["csv_path"].exists()]
if missing_inputs:
    raise FileNotFoundError("Missing input CSV(s):\n" + "\n".join(missing_inputs))

display(pd.DataFrame(DATASETS)[["label", "dt_min", "csv_path"]])
print("Outputs:", OUTPUT_ROOT.resolve())

## Analysis functions

Each cell is reindexed to its continuous integer frame range, but missing signal values remain `NaN`. Cross-correlations are calculated only from finite overlapping pairs. Group curves reproduce the pair-count weighting, baseline correction, and multi-tau smoothing used by the manuscript analysis notebook. LLI is calculated per cell from the baseline-corrected, unsmoothed curve.

In [ ]:
REQUIRED_COLUMNS = ("cell_id", "frame", "ch1_detrended", "ch2_detrended")


def load_dataset(csv_path):
    df = pd.read_csv(csv_path)
    missing = [column for column in REQUIRED_COLUMNS if column not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in {csv_path}: {missing}")
    return df


def zscore_nan(values):
    values = np.asarray(values, dtype=float)
    mean = np.nanmean(values)
    std = np.nanstd(values)
    if not np.isfinite(std) or std <= 0:
        return np.full_like(values, np.nan)
    return (values - mean) / std


def prepare_cell_series(cell_df):
    cell_df = cell_df.sort_values("frame")
    first_frame = int(cell_df["frame"].min())
    last_frame = int(cell_df["frame"].max())
    frames = np.arange(first_frame, last_frame + 1, dtype=int)
    if len(frames) < max(CONFIG["cell_min_points"], 2 * CONFIG["max_lag"] + 3):
        return None

    continuous = cell_df.set_index("frame").reindex(frames)
    sox2 = pd.to_numeric(continuous["ch1_detrended"], errors="coerce").to_numpy(float)
    mcp = pd.to_numeric(continuous["ch2_detrended"], errors="coerce").to_numpy(float)
    if CONFIG["use_zscore"]:
        sox2 = zscore_nan(sox2)
        mcp = zscore_nan(mcp)
    if np.all(~np.isfinite(sox2)) or np.all(~np.isfinite(mcp)):
        return None
    return sox2, mcp


def correlate_at_lags(x, y):
    correlation = np.full(LAGS.shape, np.nan, dtype=float)
    pair_counts = np.zeros(LAGS.shape, dtype=float)
    for index, lag in enumerate(LAGS):
        if lag < 0:
            x_overlap, y_overlap = x[:lag], y[-lag:]
        elif lag > 0:
            x_overlap, y_overlap = x[lag:], y[:-lag]
        else:
            x_overlap, y_overlap = x, y
        finite = np.isfinite(x_overlap) & np.isfinite(y_overlap)
        pair_counts[index] = finite.sum()
        if pair_counts[index] >= CONFIG["pair_thresh"]:
            correlation[index] = np.mean(x_overlap[finite] * y_overlap[finite])
    return correlation, pair_counts


def build_dataset_matrices(df):
    rows = {
        "cell_ids": [],
        "xy_series": [],
        "cc": [], "cc_w": [],
        "sox2_ac": [], "sox2_ac_w": [],
        "mcp_ac": [], "mcp_ac_w": [],
    }
    for cell_id, cell_df in df.groupby("cell_id"):
        prepared = prepare_cell_series(cell_df)
        if prepared is None:
            continue
        sox2, mcp = prepared
        cc, cc_w = correlate_at_lags(sox2, mcp)
        sox2_ac, sox2_ac_w = correlate_at_lags(sox2, sox2)
        mcp_ac, mcp_ac_w = correlate_at_lags(mcp, mcp)
        rows["cell_ids"].append(cell_id)
        rows["xy_series"].append((sox2, mcp))
        rows["cc"].append(cc)
        rows["cc_w"].append(cc_w)
        rows["sox2_ac"].append(sox2_ac)
        rows["sox2_ac_w"].append(sox2_ac_w)
        rows["mcp_ac"].append(mcp_ac)
        rows["mcp_ac_w"].append(mcp_ac_w)

    if not rows["cell_ids"]:
        raise ValueError("No valid cells remained after quality control.")
    for key in ("cc", "cc_w", "sox2_ac", "sox2_ac_w", "mcp_ac", "mcp_ac_w"):
        rows[key] = np.vstack(rows[key])
    return rows


def weighted_mean(values, weights):
    finite = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not np.any(finite):
        return np.nan
    return float(np.sum(values[finite] * weights[finite]) / np.sum(weights[finite]))


def weighted_mean_curve(matrix, weights):
    total_weight = np.nansum(weights, axis=0)
    numerator = np.nansum(matrix * weights, axis=0)
    with np.errstate(invalid="ignore", divide="ignore"):
        curve = numerator / np.maximum(total_weight, 1e-12)
    return curve, total_weight


def baseline_offset(curve, weights):
    low, high = CONFIG["base_lag_range"]
    baseline_region = (
        (np.abs(LAGS) >= low)
        & (np.abs(LAGS) <= high)
        & np.isfinite(curve)
        & (weights > 0)
    )
    if not np.any(baseline_region):
        return 0.0
    return weighted_mean(curve[baseline_region], weights[baseline_region])


def variable_bin_smooth(curve, weights):
    smoothed = np.full_like(curve, np.nan, dtype=float)
    for index, lag in enumerate(LAGS):
        window = 1
        for minimum, maximum, candidate in CONFIG["multitau_scheme"]:
            if minimum <= abs(int(lag)) <= maximum:
                window = max(1, int(candidate))
                break
        if window == 1:
            smoothed[index] = curve[index]
            continue
        half = window // 2
        low = max(0, index - half)
        high = min(len(curve), index + half + 1)
        smoothed[index] = weighted_mean(curve[low:high], weights[low:high])
    return smoothed


def corrected_group_curve(matrix, weights):
    curve, total_weight = weighted_mean_curve(matrix, weights)
    curve = curve - baseline_offset(curve, total_weight)
    if CONFIG["use_multitau_smoothing"]:
        curve = variable_bin_smooth(curve, total_weight)
    return curve, total_weight


def asymmetry_index(curve, weights):
    positive = LAGS > 0
    negative = LAGS < 0
    return weighted_mean(curve[positive], weights[positive]) - weighted_mean(
        curve[negative], weights[negative]
    )


def bootstrap_group_curve(matrix, weights, seed):
    rng = np.random.default_rng(seed)
    n_cells = matrix.shape[0]
    bootstrap_curves = np.empty((CONFIG["boot_n"], len(LAGS)), dtype=float)
    asymmetry = np.empty(CONFIG["boot_n"], dtype=float)
    for iteration in range(CONFIG["boot_n"]):
        selected = rng.integers(0, n_cells, size=n_cells)
        curve, total_weight = corrected_group_curve(matrix[selected], weights[selected])
        bootstrap_curves[iteration] = curve
        asymmetry[iteration] = asymmetry_index(curve, total_weight)
    return {
        "mean": np.nanmean(bootstrap_curves, axis=0),
        "low": np.nanpercentile(bootstrap_curves, 2.5, axis=0),
        "high": np.nanpercentile(bootstrap_curves, 97.5, axis=0),
        "A_mean": float(np.nanmean(asymmetry)),
        "A_low": float(np.nanpercentile(asymmetry, 2.5)),
        "A_high": float(np.nanpercentile(asymmetry, 97.5)),
    }


def baseline_correct_cell(curve, weights):
    return curve - baseline_offset(curve, weights)


def lead_lag_index(curve, weights):
    window = CONFIG["lli_window_frames"]
    negative = (LAGS < 0) & (LAGS >= -window)
    positive = (LAGS > 0) & (LAGS <= window)
    return weighted_mean(curve[negative], weights[negative]) - weighted_mean(
        curve[positive], weights[positive]
    )


def cell_level_metrics(matrices):
    curves = []
    lli_values = []
    r0_values = []
    zero_index = int(np.where(LAGS == 0)[0][0])
    for curve, weights in zip(matrices["cc"], matrices["cc_w"]):
        corrected = baseline_correct_cell(curve, weights)
        lli = lead_lag_index(corrected, weights)
        if not np.isfinite(lli):
            continue
        curves.append(corrected)
        lli_values.append(lli)
        r0_values.append(corrected[zero_index])
    return np.vstack(curves), np.asarray(lli_values), np.asarray(r0_values)


def bootstrap_stat_ci(values, statistic=np.nanmedian, seed=0):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    rng = np.random.default_rng(seed)
    sampled = np.empty(CONFIG["boot_n"], dtype=float)
    for iteration in range(CONFIG["boot_n"]):
        indices = rng.integers(0, len(values), size=len(values))
        sampled[iteration] = statistic(values[indices])
    return tuple(np.percentile(sampled, (2.5, 97.5)))


def bootstrap_mean_curve(curves, seed):
    rng = np.random.default_rng(seed)
    n_cells = curves.shape[0]
    sampled = np.empty((CONFIG["curve_boot_n"], len(LAGS)), dtype=float)
    for iteration in range(CONFIG["curve_boot_n"]):
        indices = rng.integers(0, n_cells, size=n_cells)
        sampled[iteration] = np.nanmean(curves[indices], axis=0)
    return {
        "mean": np.nanmean(curves, axis=0),
        "low": np.nanpercentile(sampled, 2.5, axis=0),
        "high": np.nanpercentile(sampled, 97.5, axis=0),
    }


def time_shift_null(xy_series, seed):
    rng = np.random.default_rng(seed)
    null_medians = np.full(CONFIG["perm_n"], np.nan, dtype=float)
    minimum_shift = CONFIG["max_lag"] + 1
    for iteration in range(CONFIG["perm_n"]):
        permuted_lli = []
        for sox2, mcp in xy_series:
            maximum_shift = len(mcp) - minimum_shift
            if maximum_shift <= minimum_shift:
                continue
            shift = int(rng.integers(minimum_shift, maximum_shift))
            curve, weights = correlate_at_lags(sox2, np.roll(mcp, shift))
            curve = baseline_correct_cell(curve, weights)
            lli = lead_lag_index(curve, weights)
            if np.isfinite(lli):
                permuted_lli.append(lli)
        if permuted_lli:
            null_medians[iteration] = np.nanmedian(permuted_lli)
    return null_medians


def permutation_pvalues(null_values, observed):
    null_values = np.asarray(null_values, dtype=float)
    null_values = null_values[np.isfinite(null_values)]
    denominator = len(null_values) + 1
    p_two = (np.sum(np.abs(null_values) >= abs(observed)) + 1) / denominator
    if observed >= 0:
        p_one = (np.sum(null_values >= observed) + 1) / denominator
    else:
        p_one = (np.sum(null_values <= observed) + 1) / denominator
    return float(p_two), float(p_one)


def benjamini_hochberg(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    order = np.argsort(pvalues)
    ranked = pvalues[order]
    adjusted = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    output = np.empty_like(adjusted)
    output[order] = np.clip(adjusted, 0, 1)
    return output


def add_minute_axis(axis, dt_min):
    minute_axis = axis.twiny()
    minute_axis.set_xlim(axis.get_xlim())
    ticks = axis.get_xticks()
    minute_axis.set_xticks(ticks)
    minute_axis.set_xticklabels([f"{tick * dt_min:g}" for tick in ticks])
    minute_axis.set_xlabel("Lag (min)")
    return minute_axis


def save_and_show(figure, output_path):
    figure.savefig(output_path, bbox_inches="tight")
    plt.show()
    plt.close(figure)
    print("Saved:", output_path.resolve())

## Run the SOX2 analyses

The three datasets are processed with identical frame-based parameters. `dt_min` affects only conversion of the lag axis to minutes.

In [ ]:
results = {}

for dataset in DATASETS:
    print(f"Processing {dataset['label']} ...")
    dataframe = load_dataset(dataset["csv_path"])
    matrices = build_dataset_matrices(dataframe)

    sox2_ac = bootstrap_group_curve(matrices["sox2_ac"], matrices["sox2_ac_w"], seed=123)
    mcp_ac = bootstrap_group_curve(matrices["mcp_ac"], matrices["mcp_ac_w"], seed=456)
    cross_corr = bootstrap_group_curve(matrices["cc"], matrices["cc_w"], seed=789)

    cell_curves, lli_values, r0_values = cell_level_metrics(matrices)
    lli_curve = bootstrap_mean_curve(cell_curves, seed=CONFIG["seed"])
    lli_median = float(np.nanmedian(lli_values))
    lli_low, lli_high = bootstrap_stat_ci(lli_values, seed=CONFIG["seed"])
    null_values = time_shift_null(matrices["xy_series"], seed=CONFIG["seed"])
    p_two, p_one = permutation_pvalues(null_values, lli_median)
    r0_low, r0_high = bootstrap_stat_ci(r0_values, seed=CONFIG["seed"])

    finite_r0 = r0_values[np.isfinite(r0_values)]
    r0_p = float(wilcoxon(finite_r0).pvalue) if len(finite_r0) else np.nan
    peak_lag = int(LAGS[np.nanargmax(cross_corr["mean"])])

    results[dataset["label"]] = {
        **dataset,
        "matrices": matrices,
        "sox2_ac": sox2_ac,
        "mcp_ac": mcp_ac,
        "cross_corr": cross_corr,
        "lli_curve": lli_curve,
        "lli_values": lli_values,
        "r0_values": r0_values,
        "null_values": null_values,
        "N_cells": len(matrices["cell_ids"]),
        "peak_lag_frames": peak_lag,
        "peak_lag_min": peak_lag * dataset["dt_min"],
        "LLI_median": lli_median,
        "LLI_ci_low": float(lli_low),
        "LLI_ci_high": float(lli_high),
        "perm_p_two": p_two,
        "perm_p_one": p_one,
        "r0_median": float(np.nanmedian(r0_values)),
        "r0_ci_low": float(r0_low),
        "r0_ci_high": float(r0_high),
        "r0_wilcoxon_p": r0_p,
    }

q_values = benjamini_hochberg([results[item["label"]]["perm_p_two"] for item in DATASETS])
for dataset, q_value in zip(DATASETS, q_values):
    results[dataset["label"]]["perm_q_two_BH"] = float(q_value)

print("Analysis complete.")

## Condition-level figures

For each temporal resolution, two PDFs are generated in the same format as the repository analysis notebook:

1. SOX2 and MCP autocorrelations plus their cross-correlation.
2. Baseline-corrected cross-correlation plus the time-shift permutation null for median LLI.

Each figure is displayed below and saved as a vector PDF.

In [ ]:
def plot_ac_cc(result):
    figure, axes = plt.subplots(1, 2, figsize=(10, 4))

    axis = axes[0]
    for curve, label in ((result["sox2_ac"], "SOX2/SNAP Auto"), (result["mcp_ac"], "MCP Auto")):
        axis.fill_between(LAGS, curve["low"], curve["high"], alpha=0.2)
        axis.plot(LAGS, curve["mean"], label=label)
    axis.axhline(0, color="black", linewidth=0.8, linestyle="--")
    axis.set_xlabel("Lag (frames)")
    axis.set_ylabel("Correlation (z-scored)")
    axis.set_title(f"Autocorrelations (N={result['N_cells']})")
    axis.grid(axis="y", linestyle=":", alpha=0.6)
    axis.legend()
    add_minute_axis(axis, result["dt_min"])

    axis = axes[1]
    curve = result["cross_corr"]
    axis.fill_between(LAGS, curve["low"], curve["high"], alpha=0.2)
    axis.plot(LAGS, curve["mean"], label="SOX2/SNAP vs MCP")
    axis.axhline(0, color="black", linewidth=0.8, linestyle="--")
    axis.axvline(0, color="black", linewidth=0.8, linestyle=":")
    axis.set_xlabel("Lag (frames)")
    axis.set_ylabel("Cross-correlation (z-scored)")
    axis.set_title("Cross-correlation\nNegative lag: SOX2/SNAP leads")
    axis.grid(axis="y", linestyle=":", alpha=0.6)
    axis.legend()
    add_minute_axis(axis, result["dt_min"])

    figure.suptitle(f"SOX2 ({result['display_label']} sampling)")
    figure.tight_layout()
    save_and_show(figure, OUTPUT_A / f"A_{result['label']}_AC_CC.pdf")


def plot_lli(result):
    lag_minutes = LAGS * result["dt_min"]
    figure, axes = plt.subplots(1, 2, figsize=(10, 4))

    axis = axes[0]
    curve = result["lli_curve"]
    axis.fill_between(lag_minutes, curve["low"], curve["high"], alpha=0.2)
    axis.plot(lag_minutes, curve["mean"], label="SOX2/SNAP vs MCP")
    axis.axhline(0, color="black", linewidth=0.8, linestyle="--")
    axis.axvline(0, color="black", linewidth=0.8, linestyle=":")
    axis.set_xlabel("Lag (min)")
    axis.set_ylabel("Cross-correlation (z-scored)")
    axis.set_title("Mean baseline-corrected cross-correlation\nNegative lag: SOX2/SNAP leads")
    axis.grid(axis="y", linestyle=":", alpha=0.6)
    axis.legend()

    axis = axes[1]
    finite_null = result["null_values"][np.isfinite(result["null_values"])]
    axis.hist(finite_null, bins=30, alpha=0.7)
    axis.axvline(result["LLI_median"], color="black", linewidth=1.5, label="Observed median LLI")
    axis.axvline(0, color="black", linewidth=0.8, linestyle="--")
    axis.set_xlabel("Median LLI (time-shift null)")
    axis.set_ylabel("Count")
    axis.set_title("Time-shift permutation null")
    axis.grid(axis="y", linestyle=":", alpha=0.6)
    axis.legend()

    statistics = (
        f"N={result['N_cells']}; median LLI={result['LLI_median']:.4f}; "
        f"95% CI [{result['LLI_ci_low']:.4f}, {result['LLI_ci_high']:.4f}]; "
        f"two-sided p={result['perm_p_two']:.4g}"
    )
    figure.suptitle(f"SOX2 ({result['display_label']} sampling)\n{statistics}", y=1.04)
    figure.tight_layout()
    save_and_show(figure, OUTPUT_LLI / f"LLI_{result['label']}.pdf")


for dataset in DATASETS:
    result = results[dataset["label"]]
    plot_ac_cc(result)
    plot_lli(result)

    pd.DataFrame({
        "cell_id": result["matrices"]["cell_ids"],
        "LLI": result["lli_values"],
        "r0": result["r0_values"],
    }).to_csv(OUTPUT_LLI / f"LLI_cells_{result['label']}.csv", index=False)

## Temporal-resolution comparison

The comparison figures place all three SOX2 datasets in a common figure while preserving the condition-specific sampling interval. The forest plot uses the raw LLI convention: positive values indicate a tendency for SOX2/SNAP to lead MCP.

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for axis, dataset in zip(axes, DATASETS):
    result = results[dataset["label"]]
    curve = result["cross_corr"]
    axis.fill_between(LAGS, curve["low"], curve["high"], alpha=0.2)
    axis.plot(LAGS, curve["mean"], color="tab:blue")
    axis.axhline(0, color="black", linewidth=0.8, linestyle="--")
    axis.axvline(0, color="black", linewidth=0.8, linestyle=":")
    axis.set_title(f"{result['display_label']} (N={result['N_cells']})")
    axis.set_xlabel("Lag (frames)")
    axis.grid(axis="y", linestyle=":", alpha=0.6)
    add_minute_axis(axis, result["dt_min"])
axes[0].set_ylabel("Cross-correlation (z-scored)")
figure.suptitle("SOX2-SNAP versus MCP: temporal-resolution comparison\nNegative lag: SOX2/SNAP leads", y=1.12)
figure.tight_layout()
save_and_show(figure, OUTPUT_SUMMARY / "SOX2_cross_correlation_comparison.pdf")


figure, axis = plt.subplots(figsize=(7, 3.8))
y_positions = np.arange(len(DATASETS))
for y_position, dataset in zip(y_positions, DATASETS):
    result = results[dataset["label"]]
    median = result["LLI_median"]
    low = result["LLI_ci_low"]
    high = result["LLI_ci_high"]
    axis.errorbar(
        median,
        y_position,
        xerr=[[median - low], [high - median]],
        fmt="o",
        color="black",
        capsize=3,
    )
    axis.text(
        max(high, median) + 0.015,
        y_position,
        f"N={result['N_cells']}, p={result['perm_p_two']:.3g}, q={result['perm_q_two_BH']:.3g}",
        va="center",
    )
axis.axvline(0, color="black", linewidth=1, linestyle="--")
axis.set_yticks(y_positions)
axis.set_yticklabels([dataset["display_label"] for dataset in DATASETS])
axis.invert_yaxis()
axis.set_xlim(-0.08, 0.28)
axis.set_xlabel("Median LLI with bootstrap 95% CI (positive: SOX2/SNAP leads)")
axis.set_title("SOX2 lead-lag analysis across sampling intervals")
axis.grid(axis="x", linestyle=":", alpha=0.6)
figure.tight_layout()
save_and_show(figure, OUTPUT_SUMMARY / "SOX2_LLI_forest.pdf")

In [ ]:
FONT_SIZE = 14

crosscorr_panels = [
    ("SOX2_1min", "1 min interval"),
    ("SOX2_30s", "30 s interval"),
]

lli_panels = [
    ("SOX2_30s", "SOX2      \n(30 s interval)"),
    ("SOX2_1min", "SOX2       \n(1 min interval)"),
]

figure, axes = plt.subplots(
    1,
    3,
    figsize=(15, 3.3),
    gridspec_kw={
        "width_ratios": [1.0, 1.0, 1.65],
    },
)

crosscorr_y_min = np.inf
crosscorr_y_max = -np.inf


# ------------------------------------------------------------
# Cross-correlation plots
# ------------------------------------------------------------
for axis, (label, interval_label) in zip(
    axes[:2],
    crosscorr_panels,
):
    result = results[label]
    curve = result["cross_corr"]
    lag_minutes = LAGS * result["dt_min"]

    crosscorr_y_min = min(
        crosscorr_y_min,
        np.nanmin(curve["low"]),
    )
    crosscorr_y_max = max(
        crosscorr_y_max,
        np.nanmax(curve["high"]),
    )

    axis.fill_between(
        lag_minutes,
        curve["low"],
        curve["high"],
        color="tab:blue",
        alpha=0.18,
        linewidth=0,
    )
    axis.plot(
        lag_minutes,
        curve["mean"],
        color="tab:blue",
        linewidth=2.5,
        label="SOX2",
    )

    axis.axhline(
        0,
        color="black",
        linewidth=1,
        linestyle="--",
    )
    axis.axvline(
        0,
        color="black",
        linewidth=1,
        linestyle=":",
    )

    maximum_lag_min = np.max(np.abs(lag_minutes))

    axis.set_xlim(
        -maximum_lag_min,
        maximum_lag_min,
    )
    axis.set_xticks(
        np.linspace(
            -maximum_lag_min,
            maximum_lag_min,
            5,
        )
    )

    axis.set_xlabel(
        r"Lag $\tau$ (min)",
        fontsize=FONT_SIZE,
    )
    axis.set_title(
        "Mean cross-correlation ±95% CI\n"
        f"{interval_label}",
        fontsize=FONT_SIZE,
    )

    axis.grid(
        axis="both",
        linestyle=":",
        linewidth=0.8,
        alpha=0.5,
    )
    axis.legend(
        frameon=False,
        loc="upper right",
        fontsize=FONT_SIZE,
    )
    axis.tick_params(labelsize=FONT_SIZE)

axes[0].set_ylabel(
    "Cross-corr (baseline-corrected)",
    fontsize=FONT_SIZE,
)

y_range = crosscorr_y_max - crosscorr_y_min
y_padding = 0.08 * y_range

for axis in axes[:2]:
    axis.set_ylim(
        crosscorr_y_min - y_padding,
        crosscorr_y_max + y_padding,
    )


# ------------------------------------------------------------
# Combined LLI plot
# Positive displayed LLI = MCP leads
# ------------------------------------------------------------
axis = axes[2]
y_positions = np.arange(len(lli_panels))

displayed_values = []

for y_position, (label, row_label) in zip(
    y_positions,
    lli_panels,
):
    result = results[label]

    # Raw LLI:
    #   positive = SOX2/SNAP leads
    #
    # Displayed LLI:
    #   positive = MCP leads
    displayed_median = -result["LLI_median"]
    displayed_low = -result["LLI_ci_high"]
    displayed_high = -result["LLI_ci_low"]

    displayed_values.append(
        (
            displayed_low,
            displayed_median,
            displayed_high,
        )
    )

    axis.errorbar(
        displayed_median,
        y_position,
        xerr=[
            [displayed_median - displayed_low],
            [displayed_high - displayed_median],
        ],
        fmt="o",
        markersize=6,
        color="black",
        ecolor="black",
        elinewidth=1.5,
        capsize=3,
        capthick=1.2,
    )

    axis.text(
        0.98,
        y_position,
        (
            f"n={result['N_cells']}, "
            f"p={result['perm_p_two']:.3g}, "
            f"q={result['perm_q_two_BH']:.3g}"
        ),
        transform=axis.get_yaxis_transform(),
        horizontalalignment="right",
        verticalalignment="center",
        fontsize=FONT_SIZE,
    )

axis.axvline(
    0,
    color="black",
    linewidth=1,
    linestyle="--",
)

all_lows = [value[0] for value in displayed_values]
all_highs = [value[2] for value in displayed_values]

x_min = min(-0.10, min(all_lows) - 0.03)
x_max = max(0.20, max(all_highs) + 0.12)

axis.set_xlim(x_min, x_max)
axis.set_ylim(
    len(lli_panels) - 0.5,
    -0.5,
)

axis.set_yticks(y_positions)
axis.set_yticklabels(
    [row_label for _, row_label in lli_panels],
    fontsize=FONT_SIZE,
)

axis.set_xlabel(
    "LLI (median) ± bootstrap 95% CI | "
    "positive: MCP leads",
    fontsize=FONT_SIZE,
)
axis.set_title(
    "LLI + time-shift permutation",
    fontsize=FONT_SIZE,
)

axis.grid(
    axis="x",
    linestyle=":",
    linewidth=0.8,
    alpha=0.5,
)
axis.tick_params(axis="x", labelsize=FONT_SIZE)

figure.tight_layout(w_pad=2.5)

save_and_show(
    figure,
    OUTPUT_SUMMARY
    / "SOX2_1min_30s_cross_correlation_and_LLI.pdf",
)

## Exported statistics and validation

The primary ordering test is the two-sided time-shift permutation test. The lag-zero correlation quantifies coupling and is reported separately from the LLI asymmetry test.

In [ ]:
summary_rows = []
cross_correlation_rows = []
for dataset in DATASETS:
    result = results[dataset["label"]]
    summary_rows.append({
        "condition": result["label"],
        "sampling_interval_min": result["dt_min"],
        "N_cells": result["N_cells"],
        "peak_lag_frames": result["peak_lag_frames"],
        "peak_lag_min": result["peak_lag_min"],
        "LLI_median": result["LLI_median"],
        "LLI_ci_low": result["LLI_ci_low"],
        "LLI_ci_high": result["LLI_ci_high"],
        "permutation_p_two_sided": result["perm_p_two"],
        "permutation_p_one_sided": result["perm_p_one"],
        "permutation_q_two_sided_BH": result["perm_q_two_BH"],
        "r0_median": result["r0_median"],
        "r0_ci_low": result["r0_ci_low"],
        "r0_ci_high": result["r0_ci_high"],
        "r0_wilcoxon_p": result["r0_wilcoxon_p"],
    })
    for lag, mean, low, high in zip(
        LAGS,
        result["cross_corr"]["mean"],
        result["cross_corr"]["low"],
        result["cross_corr"]["high"],
    ):
        cross_correlation_rows.append({
            "condition": result["label"],
            "lag_frames": int(lag),
            "lag_min": lag * result["dt_min"],
            "cross_correlation_mean": mean,
            "cross_correlation_ci_low": low,
            "cross_correlation_ci_high": high,
        })

summary_df = pd.DataFrame(summary_rows)
cross_correlation_df = pd.DataFrame(cross_correlation_rows)
summary_df.to_csv(OUTPUT_SUMMARY / "SOX2_temporal_resolution_summary.csv", index=False)
cross_correlation_df.to_csv(OUTPUT_SUMMARY / "SOX2_cross_correlation_curves.csv", index=False)

expected_cell_counts = {"SOX2_30s": 97, "SOX2_1min": 80, "SOX2_2min": 32}
observed_cell_counts = summary_df.set_index("condition")["N_cells"].to_dict()
assert observed_cell_counts == expected_cell_counts, (observed_cell_counts, expected_cell_counts)
assert (summary_df["peak_lag_frames"] == 0).all(), summary_df[["condition", "peak_lag_frames"]]

display(summary_df)
print("Validation passed: N=97/80/32 and the group cross-correlation peak is at lag 0 in all conditions.")
print("The LLI confidence intervals include zero; no significant two-sided lead-lag asymmetry is detected.")
print("Saved summary:", (OUTPUT_SUMMARY / "SOX2_temporal_resolution_summary.csv").resolve())

## Interpretation

The 30-s and 1-min datasets reproduce the main feature of the original 2-min analysis: the group SOX2-transcription cross-correlation is maximal at zero lag. The raw LLI values show a small tendency toward the SOX2-leading direction, but the bootstrap confidence intervals include zero and the two-sided time-shift permutation tests do not establish a systematic lead-lag asymmetry. These data do not exclude ordering faster than the 30-s sampling interval.